# Music stage: ACE-Step 1.5 score candidates

Generates 3 candidate scores for the film from `storyboard.json` (caption, bpm, length), logs every call in the same format as `runs.jsonl`, and packs everything into one zip to download.

**Run:** Runtime → Change runtime type → GPU (any: T4 is enough, the model needs under 4 GB), then Runtime → Run all.

**After:** unzip into `outputs/music/candidates/` locally, listen, copy the chosen one to `outputs/music/score.wav`, and append `music_runs.jsonl` to `runs.jsonl`.

In [ ]:
!pip -q install -U "diffusers>=0.39" transformers accelerate soundfile

In [ ]:
# The storyboard is the contract: read the music settings from the public repo, not from this notebook.
import json, urllib.request
REPO_RAW = "https://raw.githubusercontent.com/Robelabcd/heritage-short/main/storyboard.json"
storyboard = json.load(urllib.request.urlopen(REPO_RAW))
music = storyboard["music"]
duration = sum(s["duration_s"] for s in storyboard["shots"]) + storyboard["end_card"]["duration_s"] + 2
print("caption:", music["caption"])
print("bpm:", music["bpm"], "| instrumental:", music["instrumental"], "| duration:", duration, "s")

In [ ]:
import torch, time
from diffusers import AceStepPipeline

MODEL = "ACE-Step/acestep-v15-xl-turbo-diffusers"
gpu = torch.cuda.get_device_name(0)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16   # T4 has no bf16
t0 = time.time()
pipe = AceStepPipeline.from_pretrained(MODEL, torch_dtype=dtype).to("cuda")
print(f"loaded {MODEL} on {gpu} ({dtype}) in {time.time() - t0:.0f}s")

In [ ]:
import os, soundfile as sf
from datetime import datetime, timezone
from IPython.display import Audio, display

os.makedirs("music", exist_ok=True)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SEEDS = [1, 2, 3]
log = []
for seed in SEEDS:
    params = {"prompt": music["caption"], "lyrics": "[Instrumental]", "audio_duration": duration,
              "bpm": music["bpm"], "timesignature": "4", "num_inference_steps": 8}
    torch.cuda.synchronize(); t = time.time()
    audio = pipe(**params, generator=torch.Generator("cuda").manual_seed(seed)).audios
    torch.cuda.synchronize(); latency = round(time.time() - t, 3)
    path = f"music/score_seed{seed}.wav"
    sf.write(path, audio[0].T.cpu().float().numpy(), pipe.sample_rate)
    log.append({"ts": datetime.now(timezone.utc).isoformat(timespec="seconds"), "run_id": run_id,
                "stage": "music", "shot": None, "provider": f"colab ({gpu})", "model": MODEL,
                "params": params, "seed": seed, "attempt": 1, "status": "ok",
                "latency_s": latency, "gpu_minutes": round(latency / 60, 3), "cost_usd": 0.0})
    print(f"seed {seed}: {latency:.1f}s -> {path}")
    display(Audio(path))

with open("music/music_runs.jsonl", "w") as f:
    f.writelines(json.dumps(r) + "\n" for r in log)

In [ ]:
!cd music && zip -q ../music_candidates.zip *
from google.colab import files
files.download("music_candidates.zip")